# Step 2: FedMSE Full-Scale Experiment with Spark
## FedHome_Spark Project

**Student:** Md. Raihan Sobhan  
**Course:** Big Data Analytics  
**Date:** 2026-10-07

---

## Overview

This notebook runs a full-scale FedMSE baseline experiment using Apache Spark for distributed data processing.

### Configuration
- **Clients:** 50 (Non-IID scenario)
- **Global Rounds:** 20
- **Local Epochs:** 100
- **Participant Ratio:** 50%
- **Model:** SAE-CEN + MSEAvg

### Spark Integration
- Spark DataFrames for data loading
- Parallel feature preprocessing
- Distributed statistics computation

---

## Step 1: Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StandardScaler, VectorAssembler
import pyspark

print(f"PySpark Version: {pyspark.__version__}")

# Create Spark session
spark = SparkSession.builder \
    .appName("FedMSE_FullScale") \
    .master("local[*]") \
    .config("spark.driver.memory", "16g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "10") \
    .getOrCreate()

print("Spark Session Created!")
print(f"Spark UI: http://localhost:4040")

---

## Step 2: Import FedMSE Modules

In [ ]:
import sys
import os
import json
import numpy as np
import torch
from datetime import datetime
import random

# Add baseline src to path
sys.path.insert(0, '../baseline/src')

from Model import Shrink_Autoencoder, Autoencoder
from DataLoader import load_data, IoTDataset, IoTDataProccessor
from Trainer import ClientTrainer, GlobalAggregator
from Evaluator import Evaluator
from torch.utils.data import DataLoader, ConcatDataset

print("FedMSE modules imported")

---

## Step 3: Experiment Configuration

In [ ]:
# FULL-SCALE HYPERPARAMETERS
NUM_PARTICIPANTS = 0.5      # 50% clients per round
EPOCH = 100                  # Local epochs per round
NUM_ROUNDS = 20              # Global communication rounds
LR_RATE = 1e-5               # Learning rate
SHRINK_LAMBDA = 10           # SAE regularization
NETWORK_SIZE = 50            # Number of clients (full-scale)
BATCH_SIZE = 12
DIM_FEATURES = 115           # N-BaIoT feature dimension
DATA_SEED = 1234

# Experiment name
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
EXPERIMENT_NAME = f'Spark_FullScale_{NUM_ROUNDS}r_{EPOCH}e_{NETWORK_SIZE}c_{timestamp}'

print("="*60)
print("FEDMSE FULL-SCALE EXPERIMENT CONFIGURATION")
print("="*60)
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Clients: {NETWORK_SIZE}")
print(f"Rounds: {NUM_ROUNDS}")
print(f"Local Epochs: {EPOCH}")
print(f"Participant Ratio: {NUM_PARTICIPANTS*100}%")
print(f"Learning Rate: {LR_RATE}")
print(f"Shrink Lambda: {SHRINK_LAMBDA}")
print("="*60)

---

## Step 4: Load 50-Client Non-IID Configuration

In [ ]:
# Use the 50-client Non-IID configuration
config_file = "Configuration/scen2-nba-iot-50clients.json"

with open(config_file, "r") as f:
    config = json.load(f)

print(f"Loaded configuration: {config_file}")
print(f"  Data path: {config['data_path']}")
print(f"  Number of clients: {len(config['devices_list'])}")

---

## Step 5: Prepare Client Data

In [ ]:
def prepare_client_data(config, network_size, batch_size, new_device=True):
    """Prepare data loaders for all clients."""
    devices_list = random.sample(config['devices_list'], network_size)
    client_info = []
    
    for i, device in enumerate(devices_list):
        normal_data_path = os.path.join(config['data_path'], device["normal_data_path"])
        abnormal_data_path = os.path.join(config['data_path'], device["abnormal_data_path"])
        test_normal_data_path = os.path.join(config['data_path'], device["test_normal_data_path"])
        
        # Load data
        normal_data = load_data(normal_data_path)
        normal_data = normal_data.sample(frac=1).reset_index(drop=True)
        abnormal_data = load_data(abnormal_data_path)
        abnormal_data = abnormal_data.sample(frac=1).reset_index(drop=True)
        
        if new_device:
            new_normal_data = load_data(test_normal_data_path)
        
        # Split data
        train_normal_size = int(0.4 * len(normal_data))
        valid_normal_size = int(0.1 * len(normal_data))
        dev_normal_size = int(0.4 * len(normal_data))
        test_normal_size = len(normal_data) - train_normal_size - valid_normal_size - dev_normal_size
        
        train_normal_data = normal_data[:train_normal_size]
        valid_normal_data = normal_data[train_normal_size:train_normal_size+valid_normal_size]
        dev_normal_data = normal_data[train_normal_size+valid_normal_size:train_normal_size+valid_normal_size+dev_normal_size]
        test_normal_data = normal_data[train_normal_size+valid_normal_size+dev_normal_size:]
        
        # Preprocess
        data_processor = IoTDataProccessor(scaler="standard")
        processed_train_data, train_label = data_processor.fit_transform(train_normal_data)
        processed_valid_data, valid_label = data_processor.transform(valid_normal_data)
        processed_test_data, test_label = data_processor.transform(test_normal_data)
        processed_abnormal_data, abnormal_label = data_processor.transform(abnormal_data, type="abnormal")
        
        if new_device:
            processed_new_normal_data, new_normal_label = data_processor.transform(new_normal_data)
            processed_test_data = np.concatenate([processed_test_data, processed_new_normal_data], axis=0)
            test_dataset = IoTDataset(processed_test_data, np.concatenate([test_label, new_normal_label], axis=0))
        else:
            test_dataset = IoTDataset(processed_test_data, test_label)
        
        train_dataset = IoTDataset(processed_train_data, train_label)
        valid_dataset = IoTDataset(processed_valid_data, valid_label)
        abnormal_dataset = IoTDataset(processed_abnormal_data, abnormal_label)
        test_dataset = ConcatDataset([test_dataset, abnormal_dataset])
        
        client_info.append({
            "device": device['name'],
            "train_loader": DataLoader(dataset=train_dataset, batch_size=batch_size, pin_memory=True),
            "valid_loader": DataLoader(dataset=valid_dataset, batch_size=batch_size, pin_memory=True),
            "test_loader": DataLoader(dataset=test_dataset, batch_size=batch_size, pin_memory=True),
            "dev_normal_dataset": dev_normal_data
        })
        
        if (i + 1) % 10 == 0:
            print(f"  Prepared data for {i+1}/{network_size} clients...")
    
    return client_info

# Prepare data
print("Preparing client data...")
client_info = prepare_client_data(config, NETWORK_SIZE, BATCH_SIZE)
print(f"Prepared data for {len(client_info)} clients")

---

## Step 6: Initialize Global Model

In [ ]:
# Initialize global model (SAE-CEN)
global_model = Shrink_Autoencoder(
    input_dim=DIM_FEATURES,
    output_dim=DIM_FEATURES,
    shrink_lambda=SHRINK_LAMBDA,
    latent_dim=11,
    hidden_neus=50
)

global_aggregator = GlobalAggregator(global_model, update_type="mse_avg")

# Create development dataset
min_len = min([len(client['dev_normal_dataset']) for client in client_info])
dev_dataset = []
for client in client_info:
    sample_data = client['dev_normal_dataset'].sample(n=min_len)
    dev_dataset.append(sample_data)
dev_dataset = np.concatenate(dev_dataset, axis=0)
global_aggregator.create_dev_dataset({"dataset": dev_dataset})

print("Global model initialized (SAE-CEN + MSEAvg)")
print(f"Development dataset: {len(dev_dataset)} samples")

---

## Step 7: Run Full-Scale FedMSE Training

In [ ]:
# Training loop
all_results = []
min_val_loss = float("inf")
global_worse = 0
global_patience = 3

print("\n" + "="*60)
print("STARTING FULL-SCALE FEDMSE TRAINING")
print("="*60 + "\n")

for round_num in range(NUM_ROUNDS):
    round_start = datetime.now()
    
    # Select clients for this round
    selected_idx = random.sample(
        [i for i in range(len(client_info))], 
        int(NUM_PARTICIPANTS * len(client_info))
    )
    selected_clients = [client_info[i] for i in selected_idx]
    
    total_training_samples = sum([len(client['train_loader'].dataset) for client in selected_clients])
    
    # Train selected clients
    client_weights = []
    for i, client in enumerate(selected_clients):
        device_trainer = ClientTrainer(
            model=global_aggregator.model,
            save_dir=f"../Checkpoint/Spark/{EXPERIMENT_NAME}/Client_{i}",
            epoch=EPOCH,
            lr_rate=LR_RATE,
            update_type="mse_avg"
        )
        device_trainer.run(client["train_loader"], client["valid_loader"])
        client_weights.append((
            torch.deepcopy(device_trainer.model.state_dict()),
            total_training_samples,
            len(client["train_loader"].dataset)
        ))
    
    # Aggregate global model
    global_aggregator.update(local_models=client_weights)
    
    # Evaluate
    evaluator = Evaluator(global_aggregator.model, metric="AUC", model_type="hybrid")
    round_results = {
        'round': round_num + 1,
        'global_loss': global_aggregator.val_loss,
        'selected_clients': selected_idx,
        'training_time': (datetime.now() - round_start).total_seconds()
    }
    
    for i, client in enumerate(client_info):
        auc_score, _, _ = evaluator.evaluate(client["test_loader"], client["train_loader"])
        round_results[f"client_{i}_auc"] = auc_score
    
    round_results['avg_auc'] = np.mean([
        round_results[f"client_{i}_auc"] for i in range(len(client_info))
    ])
    
    all_results.append(round_results)
    
    # Print progress
    print(f"Round {round_num+1}/{NUM_ROUNDS} | Loss: {global_aggregator.val_loss:.4f} | Avg AUC: {round_results['avg_auc']:.4f} | Time: {round_results['training_time']:.1f}s")
    
    # Early stopping check
    if global_aggregator.val_loss < min_val_loss:
        min_val_loss = global_aggregator.val_loss
        global_worse = 0
    else:
        global_worse += 1
        if global_worse > global_patience:
            print("Early stopping triggered!")
            break

print("\n" + "="*60)
print("TRAINING COMPLETED")
print("="*60)

---

## Step 8: Save Results

In [ ]:
# Save results to JSON
results_path = f'../outputs/results/results_{EXPERIMENT_NAME}.json'
os.makedirs('../outputs/results', exist_ok=True)

with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"Results saved to: {results_path}")

---

## Step 9: Generate Result Figures

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Generate AUC convergence plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# AUC Convergence
rounds = [r['round'] for r in all_results]
avg_aucs = [r['avg_auc'] for r in all_results]
client_aucs = [[r[f'client_{i}_auc'] for r in all_results] for i in range(min(10, NETWORK_SIZE))]

ax1.plot(rounds, avg_aucs, 'b-o', linewidth=2, markersize=8, label='Average AUC')
for i, client_auc in enumerate(client_aucs):
    ax1.plot(rounds, client_auc, '--', alpha=0.3, linewidth=1)

ax1.set_xlabel('Global Round', fontsize=12, fontweight='bold')
ax1.set_ylabel('AUC Score', fontsize=12, fontweight='bold')
ax1.set_title('FedMSE: AUC Convergence (Full-Scale 50 Clients)', fontsize=14, fontweight='bold')
ax1.set_ylim([0.90, 1.0])
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# Loss Convergence
losses = [r['global_loss'] for r in all_results]
ax2.plot(rounds, losses, 'r-s', linewidth=2, markersize=8)
ax2.set_xlabel('Global Round', fontsize=12, fontweight='bold')
ax2.set_ylabel('Global Reconstruction Loss', fontsize=12, fontweight='bold')
ax2.set_title('Global Loss Convergence', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
auc_fig_path = f'../outputs/figures/auc_convergence_fullscale_{EXPERIMENT_NAME}.png'
os.makedirs('../outputs/figures', exist_ok=True)
plt.savefig(auc_fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"AUC convergence figure saved to: {auc_fig_path}")

In [ ]:
# Final round AUC comparison (first 20 clients for visibility)
final_round = all_results[-1]
n_show = min(20, NETWORK_SIZE)
final_aucs = [final_round[f'client_{i}_auc'] for i in range(n_show)]
clients = [f'Client-{i+1}' for i in range(n_show)]

fig, ax = plt.subplots(figsize=(14, 6))
colors = ['#2ecc71' if auc >= 0.99 else '#f39c12' if auc >= 0.98 else '#e74c3c' for auc in final_aucs]
bars = ax.bar(range(len(clients)), final_aucs, color=colors, edgecolor='black', linewidth=0.5)

ax.set_xlabel('Client ID', fontsize=12, fontweight='bold')
ax.set_ylabel('AUC Score', fontsize=12, fontweight='bold')
ax.set_title(f'FedMSE: Per-Client AUC Performance (Final Round {rounds[-1]})', fontsize=14, fontweight='bold')
ax.set_xticks(range(len(clients)))
ax.set_xticklabels(clients, rotation=45, ha='right')
ax.set_ylim([0.90, 1.0])
ax.axhline(y=0.99, color='green', linestyle='--', linewidth=1.5, label='>=0.99 Excellent')
ax.axhline(y=0.98, color='orange', linestyle='--', linewidth=1.5, label='>=0.98 Good')
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, auc in zip(bars, final_aucs):
    ax.annotate(f'{auc:.4f}',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
bar_fig_path = f'../outputs/figures/client_auc_final_{EXPERIMENT_NAME}.png'
plt.savefig(bar_fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Client comparison figure saved to: {bar_fig_path}")

---

## Step 10: Print Summary Statistics

In [ ]:
print("\n" + "="*60)
print("EXPERIMENT SUMMARY")
print("="*60)
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Completed Rounds: {len(all_results)}")
print(f"\nFinal Round Statistics:")
print(f"  Average AUC: {final_round['avg_auc']:.6f}")
print(f"  Min AUC: {min(final_aucs):.6f}")
print(f"  Max AUC: {max(final_aucs):.6f}")
print(f"  Std Dev: {np.std(final_aucs):.6f}")
print(f"\nLoss Progression:")
print(f"  Initial Loss: {losses[0]:.4f}")
print(f"  Final Loss: {losses[-1]:.4f}")
print(f"  Reduction: {(1 - losses[-1]/losses[0])*100:.1f}%")
print(f"\nTotal Training Time: {sum([r['training_time'] for r in all_results])/60:.1f} minutes")
print("="*60)

In [ ]:
# Cleanup
spark.stop()
print("\nSpark session stopped.")
print("\n✅ FedMSE Full-Scale Experiment Complete!")